from utils import setup_korean_font
setup_korean_font()
# 01 전처리 Ablation — Phase 2

**분류기 고정:** Logistic Regression L2 (C=1.0)  
**데이터:** `labeled_data.csv` (7,996행, 25개 유효 변수, 불량률 0.89%)  
**CV:** Stratified 5-fold (fold_0~4.npy 공유)  
**그리드:** Scaler 2종 × 불균형 처리 5종 = 10 조합  

결측·스케일링·SMOTE는 fold 내부에서만 fit한다 (데이터 누수 방지).

In [ ]:
# Cell 1 — imports + seed
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product as iproduct

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_recall_curve, RocCurveDisplay, PrecisionRecallDisplay,
)

from utils import set_seed
from data import load_raw, get_fold, generate_splits
from preprocess import get_feature_cols, fit_transform_fold

set_seed(42)

FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR  = PROJECT_ROOT / 'results' / 'tables'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Setup OK')

## 데이터 로드 + 변수 확인

In [ ]:
df = load_raw('labeled_data')
FEAT_COLS = get_feature_cols(df)
X = df[FEAT_COLS].values
y = df['PassOrFail'].values

generate_splits(X, y)  # idempotent — skips if already exists

print(f'Rows: {len(df):,}   Features: {len(FEAT_COLS)}   Fail%: {y.mean()*100:.2f}%')
print(f'Features: {FEAT_COLS}')
print(f'\nClass balance -> 양품:{(y==0).sum():,}  불량:{(y==1).sum():,}')

## Ablation 결과 로드 (사전 계산 완료)

In [ ]:
abl_df = pd.read_csv(TABLES_DIR / 'preproc_ablation.csv')

display_cols = ['scaler', 'resample',
                'roc_auc_mean', 'roc_auc_std',
                'pr_auc_mean',  'pr_auc_std',
                'f1_mean',      'f1_std']

print('=== 전처리 Ablation 결과 (PR-AUC 기준 정렬) ===')
print(abl_df[display_cols].to_string(index=False))

best_pr  = abl_df.loc[abl_df['pr_auc_mean'].idxmax()]
best_roc = abl_df.loc[abl_df['roc_auc_mean'].idxmax()]
print(f'\nBest PR-AUC : {best_pr["scaler"]} x {best_pr["resample"]}  '
      f'PR-AUC={best_pr["pr_auc_mean"]:.4f}+/-{best_pr["pr_auc_std"]:.4f}')
print(f'Best ROC-AUC: {best_roc["scaler"]} x {best_roc["resample"]}  '
      f'ROC-AUC={best_roc["roc_auc_mean"]:.4f}+/-{best_roc["roc_auc_std"]:.4f}')

## Heatmap: Scaler × Resample

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, metric, title in [
    (axes[0], 'roc_auc_mean', 'ROC-AUC'),
    (axes[1], 'pr_auc_mean',  'PR-AUC'),
]:
    pivot = abl_df.pivot(index='resample', columns='scaler', values=metric)
    sns.heatmap(
        pivot, ax=ax, annot=True, fmt='.4f',
        cmap='YlOrRd', linewidths=0.5, linecolor='gray',
        cbar_kws={'shrink': 0.8},
    )
    ax.set_title(f'{title} (5-fold 평균)', fontsize=12)
    ax.set_xlabel('Scaler')
    ax.set_ylabel('Imbalance Strategy')

plt.suptitle('전처리 Ablation — LR-L2 고정 (C=1.0)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'preproc_ablation_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved: results/figures/preproc_ablation_heatmap.png')

StandardScaler가 전 조합에서 RobustScaler를 압도했다. Robust+None의 ROC-AUC=0.34는 saga solver가 스케일 불균형 데이터에서 수렴에 실패한 결과로, StandardScaler를 확정한다. 불균형 처리 효과는 ROC-AUC 기준 SMOTE(0.9311) > class_weight(0.9261) > None(0.9076) > ADASYN(0.9155) > undersample(0.9064) 순이다.

## ROC-AUC vs PR-AUC 산점도 + 표준편차 에러바

In [ ]:
RESAMPLE_COLORS = {
    'none':         '#4C72B0',
    'class_weight': '#DD8452',
    'smote':        '#55A868',
    'adasyn':       '#C44E52',
    'undersample':  '#8172B2',
}
SCALER_MARKERS = {'standard': 'o', 'robust': 's'}

fig, ax = plt.subplots(figsize=(9, 6))

for _, row in abl_df.iterrows():
    ax.errorbar(
        row['roc_auc_mean'], row['pr_auc_mean'],
        xerr=row['roc_auc_std'], yerr=row['pr_auc_std'],
        fmt=SCALER_MARKERS[row['scaler']],
        color=RESAMPLE_COLORS[row['resample']],
        markersize=10, capsize=4, linewidth=1.2,
        label=f"{row['scaler']}+{row['resample']}",
    )

ax.set_xlabel('ROC-AUC (5-fold 평균)', fontsize=12)
ax.set_ylabel('PR-AUC (5-fold 평균)', fontsize=12)
ax.set_title('전처리 조합별 ROC-AUC vs PR-AUC (에러바=1std)', fontsize=12)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'preproc_ablation_scatter.png', bbox_inches='tight')
plt.show()
print('Saved: results/figures/preproc_ablation_scatter.png')

standard×smote는 ROC-AUC 0.9311로 최고 성능이지만, PR-AUC는 standard×none(0.2666, std=0.1145)보다 낮다. 불균형 처리를 적용하면 PR-AUC 절댓값은 하락하지만 분산이 절반 이하로 줄어 fold 간 신뢰도가 높아진다 — 산업 운용 관점에서 안정성이 더 중요하다.

## Best 조합 PR 곡선 상세 분석

In [ ]:
# top-3 조합의 fold별 PR 곡선 비교
TOP_COMBOS = [
    ('standard', 'smote'),
    ('standard', 'class_weight'),
    ('standard', 'none'),
]
COMBO_COLORS = ['#55A868', '#DD8452', '#4C72B0']

fig, ax = plt.subplots(figsize=(8, 6))

for (scaler_name, resample_name), color in zip(TOP_COMBOS, COMBO_COLORS):
    # 5 fold 평균 PR 곡선
    all_prec, all_rec = [], []
    for fold_i in range(5):
        X_tr, X_va, y_tr, y_va = get_fold(fold_i, X, y)
        X_tr_s, X_va_s, y_tr_r = fit_transform_fold(
            X_tr, X_va, y_tr, scaler_name, resample_name
        )
        cw = 'balanced' if resample_name == 'class_weight' else None
        clf = LogisticRegression(
            penalty='l2', C=1.0, class_weight=cw,
            solver='saga', max_iter=2000, random_state=42,
        )
        clf.fit(X_tr_s, y_tr_r)
        y_prob = clf.predict_proba(X_va_s)[:, 1]
        prec, rec, _ = precision_recall_curve(y_va, y_prob)
        all_prec.append(prec)
        all_rec.append(rec)

    # 공통 recall 그리드에 보간 후 평균
    rec_grid = np.linspace(0, 1, 100)
    interp_precs = []
    for prec, rec in zip(all_prec, all_rec):
        interp_precs.append(np.interp(rec_grid, rec[::-1], prec[::-1]))
    mean_prec = np.mean(interp_precs, axis=0)
    std_prec  = np.std(interp_precs, axis=0)

    label = f'{scaler_name}+{resample_name}'
    ax.plot(rec_grid, mean_prec, color=color, label=label, linewidth=2)
    ax.fill_between(rec_grid, mean_prec - std_prec, mean_prec + std_prec,
                    color=color, alpha=0.15)

# 기준선 (무작위 분류기)
baseline_pr = y.mean()
ax.axhline(baseline_pr, color='gray', linestyle='--', linewidth=1,
           label=f'Random ({baseline_pr:.3f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Top-3 전처리 조합 PR 곡선 (5-fold 평균 ± 1std)', fontsize=12)
ax.legend(fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'preproc_ablation_pr_curves.png', bbox_inches='tight')
plt.show()
print('Saved: results/figures/preproc_ablation_pr_curves.png')

Recall 0~0.8 구간에서 standard+smote가 가장 높고 일관된 Precision을 유지한다. standard+none은 Recall이 0.5를 넘으면 Precision이 급격히 하락해 고재현율 운용점에서 신뢰할 수 없다. 이 PR 곡선 비교로 Phase 3 기본 전처리를 StandardScaler + SMOTE로 확정한다.

## 가이드북 비교 + 결론

In [ ]:
print('='*65)
print('Phase 2 전처리 Ablation 결론')
print('='*65)

best_smote = abl_df[(abl_df['scaler']=='standard') & (abl_df['resample']=='smote')].iloc[0]
best_none  = abl_df[(abl_df['scaler']=='standard') & (abl_df['resample']=='none')].iloc[0]

print(f"""
[핵심 발견]
1. StandardScaler >> RobustScaler (전 조합에서 일관)
   - Robust+None ROC-AUC=0.3406: LR의 saga solver가 RobustScaler 후
     극단적 불균형 데이터에서 수렴 실패. Phase 3부터 StandardScaler 확정.

2. 불균형 처리 효과 (ROC-AUC 기준, standard scaler):
   SMOTE({best_smote['roc_auc_mean']:.4f}) > class_weight(0.9261) > none({best_none['roc_auc_mean']:.4f}) > ADASYN(0.9155) > undersample(0.9064)

3. PR-AUC 기준: none(0.2666, std=0.1145) > smote({best_smote['pr_auc_mean']:.4f}, std={best_smote['pr_auc_std']:.4f})
   → none의 높은 PR-AUC는 분산이 커 불안정. SMOTE가 안정적.

4. 가이드북 비교:
   - 가이드북 §2.3은 전처리 옵션 비교 없이 단일 표준화만 적용 후 AE/SVM/DNN 진행.
   - 우리 ablation: 불균형 처리가 ROC-AUC를 최대 +0.024 개선 (none→SMOTE).
   - 이 단계가 '가이드북에 빠진 단계'임을 결과표에 명시.

[Phase 3 확정 전처리]
  Scaler   : StandardScaler
  Resample : SMOTE (random_state=42, k_neighbors=5)
  근거     : ROC-AUC 최고 + PR-AUC 안정적 (std 절반 이하)
""")

`StandardScaler + SMOTE` 조합을 Phase 3 이후의 기본 전처리로 확정한다. 가이드북이 생략한 클래스 불균형 처리 ablation이 ROC-AUC를 최대 +0.024 개선함을 5-fold 교차검증으로 정량 확인했다.